In [52]:
import talib
import scipy.stats
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.subplots as sub
import quantstats as qs
from jup.helpers.charts import (
    get_layout,
    get_price_trace,
    add_trade_markers,
    remove_x_gaps,
    detect_range_rule,
)
from jup.helpers.data_sources import get_exante_data, resample_ohlc

pd.options.display.width = 150

files = [
  "../res/OIH.ARCA_ChannelBreakout_300-stats.csv",
  "../res/OIH.ARCA_ChannelBreakout_1000-stats.csv", # похоже на AMZA, но сильно хуже
  "../res/COPX.ARCA_ChannelBreakout_600-stats.csv",
  "../res/ARKK.ARCA_ChannelBreakout_700-stats.csv",
  "../res/AMZA.ARCA_ChannelBreakout_500-stats.csv",
  "../res/EMQQ.ARCA_ChannelBreakout_400-stats.csv",
  "../res/BLOK.ARCA_ChannelBreakout_1000-stats.csv",
  "../res/VCR.ARCA_ChannelBreakout_1000-stats.csv", # похоже на всё
  "../res/URA.ARCA_ChannelBreakout_500-stats.csv",
  "../res/VDE.ARCA_ChannelBreakout_400-stats.csv",  # слишком похоже на AMZA
  "../res/XSD.ARCA_ChannelBreakout_900-stats.csv",  # похоже на ROBO, но хуже
  "../res/CQQQ.ARCA_ChannelBreakout_1200-stats.csv",# похоже на EMQQ, адский период
  "../res/ROBO.ARCA_ChannelBreakout_900-stats.csv",
]

# Date,Value,Drawdown,Equity,RelEquity,Change

all_data = pd.DataFrame()

for i, name in enumerate(files):
    df = pd.read_csv(name, index_col="Date")
    df.index = pd.to_datetime(df.index)
    df = df.resample("1d").pad()
    df.dropna(inplace=True)

    symbol = name[7:11].strip(".") + "-" + name[-14:-9].strip("-").strip("_")
    all_data[symbol] = df["Value"]

# import matplotlib.pyplot as plt
# f = plt.figure(figsize=(19, 15))
# plt.matshow(all_data.corr(), fignum=f.number)
# plt.xticks(range(all_data.select_dtypes(['number']).shape[1]), all_data.select_dtypes(['number']).columns, fontsize=12)
# plt.yticks(range(all_data.select_dtypes(['number']).shape[1]), all_data.select_dtypes(['number']).columns, fontsize=12)
# cb = plt.colorbar()
# cb.ax.tick_params(labelsize=15)


In [91]:
from pypfopt import EfficientFrontier
from pypfopt import risk_models
from pypfopt import expected_returns
from pypfopt import discrete_allocation

df = all_data.copy(deep=True)

# Calculate expected returns and sample covariance
# mu = expected_returns.mean_historical_return(df)
mu = expected_returns.ema_historical_return(df, span=200)
S = risk_models.sample_cov(df)

# Optimize for maximal Sharpe ratio
ef = EfficientFrontier(mu, S)
raw_weights = ef.max_sharpe()
cleaned_weights = ef.clean_weights()

for w in cleaned_weights.items():
    print(f"{w[0]:<10} {(w[1] * 100):5.1f}")

print()
print(ef.portfolio_performance(verbose=True))


OIH-300      0.0
OIH-1000     0.0
COPX-600    45.4
ARKK-700    23.5
AMZA-500     0.0
EMQQ-400     3.4
BLOK-1000    0.0
VCR-1000     0.0
URA-500     25.2
VDE-400      0.0
XSD-900      0.0
CQQQ-1200    2.6
ROBO-900     0.0

Expected annual return: 37.9%
Annual volatility: 14.9%
Sharpe Ratio: 2.41
(0.3786393865684622, 0.14874247323828252, 2.411143090204833)


RuntimeError: This event loop is already running